
# 流水特征聚合（按 id）
> 文件名：`statement_procssing.ipynb`  
> 作用：将 `train_bank_statement.csv / testaa_bank_statement.csv` 聚合为按 id 的**简洁稳健**特征，避免复杂时间窗，严格防泄漏。

**要点**
- 时间统一转为“距离基准日的天数”（基准日固定为 `2025-08-31`），保证复现性；
- 仅做简单统计（总收入/支出、笔数、活跃天数、频次、首末交易距今天数、收入/支出均值/中位数/方差、收入与支出比等）；
- 对无流水的 id 自动 0 填充并新增 `stm_missing` 标记；
- **不使用 label 信息**参与任何聚合，避免数据泄漏。


In [ ]:

# ============ 可配置区 ============
REF_DATE_STR = "2025-08-31"              # 固定参考日期（Asia/Tokyo 当地日期）
TRAIN_STM = "train_bank_statement.csv"   # 训练端流水
TEST_STM  = "testaa_bank_statement.csv"  # 测试端流水（可缺失）
IDS_FROM  = "train.csv"                  # 用于取全量 id（如果存在则对缺失流水的 id 也输出特征）

OUT_TRAIN = "train_statement_feature.csv"
OUT_TEST  = "testaa_statement_feature.csv"
# ==================================



In [ ]:

import pandas as pd
import numpy as np

REF_DATE = pd.Timestamp(REF_DATE_STR)

def _safe_to_datetime_from_unix(s):
    # 输入是秒级 unix 时间戳
    return pd.to_datetime(s, unit='s', utc=True, errors='coerce').dt.tz_convert(None)

def _aggregate_one(df):
    # df 为单个 id 的流水子表（按时间升序排序）
    if df.empty:
        return {}

    # 仅保留方向取值 {0,1}：0=收入, 1=支出
    df = df[df['direction'].isin([0,1])].copy()
    if df.empty:
        return {}

    df['date'] = df['time'].dt.date
    df['is_income'] = (df['direction'] == 0).astype(int)
    df['is_expense'] = (df['direction'] == 1).astype(int)

    total_tx_count = len(df)
    income_mask = df['is_income'] == 1
    expense_mask = df['is_expense'] == 1

    total_income = df.loc[income_mask, 'amount'].sum()
    total_expense = df.loc[expense_mask, 'amount'].sum()
    income_count = int(income_mask.sum())
    expense_count = int(expense_mask.sum())
    net_income = total_income - total_expense

    active_days = df['date'].nunique()
    tx_frequency = total_tx_count / max(active_days, 1)

    last_tx = df['time'].max()
    first_tx = df['time'].min()
    last_tx_days_ago = (REF_DATE - last_tx.normalize()).days
    first_tx_days_ago = (REF_DATE - first_tx.normalize()).days
    span_days = (last_tx.normalize() - first_tx.normalize()).days

    # 收/支统计
    avg_income = df.loc[income_mask, 'amount'].mean() if income_count > 0 else 0.0
    income_std = df.loc[income_mask, 'amount'].std(ddof=0) if income_count > 1 else 0.0
    median_income = df.loc[income_mask, 'amount'].median() if income_count > 0 else 0.0

    avg_expense = df.loc[expense_mask, 'amount'].mean() if expense_count > 0 else 0.0
    expense_std = df.loc[expense_mask, 'amount'].std(ddof=0) if expense_count > 1 else 0.0
    median_expense = df.loc[expense_mask, 'amount'].median() if expense_count > 0 else 0.0

    # “大额收入”：相对该 id 自身中位数判断
    big_income_count = int((df.loc[income_mask, 'amount'] > median_income).sum()) if income_count > 0 else 0
    big_income_ratio = big_income_count / max(income_count, 1)

    income_expense_ratio = total_income / (total_expense + 1e-6)

    return {
        'total_income': total_income,
        'total_expense': total_expense,
        'net_income': net_income,
        'income_count': income_count,
        'expense_count': expense_count,
        'total_tx_count': total_tx_count,
        'active_days': active_days,
        'tx_frequency': tx_frequency,
        'last_tx_days_ago': last_tx_days_ago,
        'first_tx_days_ago': first_tx_days_ago,
        'span_days': span_days,
        'income_expense_ratio': income_expense_ratio,
        'avg_income': 0.0 if pd.isna(avg_income) else float(avg_income),
        'income_std': 0.0 if pd.isna(income_std) else float(income_std),
        'median_income': 0.0 if pd.isna(median_income) else float(median_income),
        'avg_expense': 0.0 if pd.isna(avg_expense) else float(avg_expense),
        'expense_std': 0.0 if pd.isna(expense_std) else float(expense_std),
        'median_expense': 0.0 if pd.isna(median_expense) else float(median_expense),
        'big_income_count': big_income_count,
        'big_income_ratio': big_income_ratio,
    }

def build_statement_features(path_csv, ids_hint=None):
    usecols = ['id','time','direction','amount']
    df = pd.read_csv(path_csv, usecols=usecols)
    df['time'] = _safe_to_datetime_from_unix(df['time'])
    df = df.dropna(subset=['id','time','direction','amount'])
    df = df.sort_values(['id','time'])

    rows = []
    for gid, part in df.groupby('id', sort=False):
        a = _aggregate_one(part.copy())
        if a is None:
            continue
        a['id'] = gid
        rows.append(a)

    feat_df = pd.DataFrame(rows)
    if ids_hint is not None:
        ids = pd.DataFrame({'id': ids_hint}).drop_duplicates()
        feat_df = ids.merge(feat_df, on='id', how='left')

    empty_cols = [c for c in feat_df.columns if c != 'id']
    feat_df['stm_missing'] = feat_df[empty_cols].isna().all(axis=1).astype(int)
    feat_df[empty_cols] = feat_df[empty_cols].fillna(0)

    for c in ['income_expense_ratio','tx_frequency','big_income_ratio']:
        if c in feat_df.columns:
            feat_df[c] = feat_df[c].clip(lower=0, upper=1e6)

    return feat_df


In [3]:

# 训练端生成
ids_hint = None
import os
if os.path.exists(IDS_FROM):
    try:
        ids_hint = pd.read_csv(IDS_FROM, usecols=['id'])['id'].values
    except Exception:
        ids_hint = None

if os.path.exists(TRAIN_STM):
    tr_feat = build_statement_features(TRAIN_STM, ids_hint=ids_hint)
    tr_feat.to_csv(OUT_TRAIN, index=False, encoding='utf-8')
    print(f"[OK] 保存训练流水特征 -> {OUT_TRAIN}  shape={tr_feat.shape}")
    display(tr_feat.head())
else:
    print(f"[WARN] 未找到 {TRAIN_STM}，跳过训练流水特征生成。")


[OK] 保存训练流水特征 -> train_statement_feature.csv  shape=(53480, 22)


,id,total_income,total_expense,net_income,income_count,expense_count,total_tx_count,active_days,tx_frequency,last_tx_days_ago,...,income_expense_ratio,avg_income,income_std,median_income,avg_expense,expense_std,median_expense,big_income_count,big_income_ratio,stm_missing
0,0,59707.50,12079.50,47628.00,6.0,42.0,48.0,41.0,1.170732,6000.0,...,4.942878,9951.250000,589.697531,10081.400,287.607143,266.070141,257.250,3.0,0.5,0
1,1,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000,0.000000,0.000000,0.000,0.0,0.0,1
2,2,6522.38,15883.72,-9361.34,4.0,44.0,48.0,46.0,1.043478,4313.0,...,0.410633,1630.595000,75.244522,1659.650,360.993636,592.533664,227.175,2.0,0.5,0
3,3,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000,0.000000,0.000000,0.000,0.0,0.0,1
4,4,20339.90,30823.10,-10483.20,6.0,87.0,93.0,83.0,1.120482,5999.0,...,0.659891,3389.983333,190.526025,3356.675,354.288506,553.705960,243.500,3.0,0.5,0


In [ ]:

# 测试端生成（如果有测试流水）
import os
if os.path.exists(TEST_STM):
    te_feat = build_statement_features(TEST_STM, ids_hint=None)
    te_feat.to_csv(OUT_TEST, index=False, encoding='utf-8')
    print(f"[OK] 保存测试流水特征 -> {OUT_TEST}  shape={te_feat.shape}")
    display(te_feat.head())
else:
    print(f"[WARN] 未找到 {TEST_STM}，跳过测试流水特征生成。")
 

[OK] 保存测试流水特征 -> testaa_statement_feature.csv  shape=(8180, 22)


,total_income,total_expense,net_income,income_count,expense_count,total_tx_count,active_days,tx_frequency,last_tx_days_ago,first_tx_days_ago,...,avg_income,income_std,median_income,avg_expense,expense_std,median_expense,big_income_count,big_income_ratio,id,stm_missing
0,50570.00,11245.52,39324.48,5,41,46,43,1.069767,8140,8305,...,10114.000000,591.594794,10146.000,274.280976,197.697990,242.96,2,0.400000,53480,0
1,18393.90,20161.94,-1768.04,3,53,56,49,1.142857,6780,6924,...,6131.300000,238.743541,6076.200,380.413962,598.840611,259.57,1,0.333333,53481,0
2,36612.34,44977.46,-8365.12,6,68,74,66,1.121212,5570,5742,...,6102.056667,534.617623,6158.540,661.433235,909.002935,358.55,3,0.500000,53482,0
3,24156.80,24048.52,108.28,6,67,73,63,1.158730,6491,6650,...,4026.133333,174.579520,4013.000,358.933134,565.959073,294.39,3,0.500000,53483,0
4,48315.75,29797.86,18517.89,6,85,91,81,1.123457,6275,6449,...,8052.625000,346.945984,8117.625,350.563059,423.918739,272.90,3,0.500000,53484,0
